In [24]:
# ============================================================
# DriveSmart — Phase 1: Web Scraping
# Location: San Jose, California
# Dates: 2026-03-22 to 2026-03-29
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import time
import csv
import os
import random
import glob
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.safari.options import Options as SafariOptions


# ── 2. Configuration ─────────────────────────────────────────
LOCATION           = "San Jose, California, United States"
START_DATE_STR     = "2026-03-22"
END_DATE_STR       = "2026-03-29"
BROWSER            = "chrome"
HEADLESS           = False
DELAY_BETWEEN_DAYS = 6
MAX_RETRIES        = 3
OUTPUT_DIR         = "kayak_data"

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) "
    "Gecko/20100101 Firefox/125.0",
]


# ── 3. Driver setup ──────────────────────────────────────────
def setup_driver(browser=BROWSER, headless=HEADLESS):
    ua = random.choice(USER_AGENTS)
    browser = browser.lower()

    if browser == "chrome":
        opts = ChromeOptions()
        if headless:
            opts.add_argument("--headless=new")
        opts.add_argument(f"user-agent={ua}")
        opts.add_argument("--disable-blink-features=AutomationControlled")
        opts.add_experimental_option("excludeSwitches", ["enable-automation"])
        opts.add_experimental_option("useAutomationExtension", False)
        opts.add_argument("--window-size=1400,900")
        # FIX: suppress Google One-Tap / Sign-in prompts at the browser level
        opts.add_argument("--disable-features=IdentityConsistencyConsentBump")
        opts.add_argument("--disable-features=GoogleOneTapLogin")
        prefs = {
            "credentials_enable_service": False,
            "profile.password_manager_enabled": False,
        }
        opts.add_experimental_option("prefs", prefs)
        return webdriver.Chrome(options=opts)

    elif browser == "firefox":
        opts = FirefoxOptions()
        if headless:
            opts.add_argument("--headless")
        opts.set_preference("general.useragent.override", ua)
        opts.set_preference("dom.webdriver.enabled", False)
        return webdriver.Firefox(options=opts)

    elif browser == "safari":
        opts = SafariOptions()
        opts.set_capability("safari:automaticInspection", True)
        opts.set_capability("safari:automaticProfiling", True)
        return webdriver.Safari(options=opts)

    else:
        raise ValueError(f"Unsupported browser: '{browser}'.")


# ── 4. Date utilities ────────────────────────────────────────
def get_date_range(start_date_str, end_date_str):
    start = datetime.strptime(start_date_str, "%Y-%m-%d")
    end   = datetime.strptime(end_date_str,   "%Y-%m-%d")
    dates, current = [], start
    while current <= end:
        dates.append(current)
        current += timedelta(days=1)
    return dates

def format_kayak_aria(date_obj):
    """e.g. 'Sunday, March 22, 2026'"""
    return date_obj.strftime("%A, %B %-d, %Y")   # Windows: replace %-d with %#d

def format_kayak_display(date_obj):
    """e.g. 'Sun 3/22'"""
    return date_obj.strftime("%a %-m/%-d")        # Windows: replace %-d with %#d


# ── 5. Popup / overlay handling ──────────────────────────────
def handle_initial_popups(driver):
    """Dismiss the KAYAK cookie/consent banner if present."""
    try:
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "button[class*='RxNS-mod-theme-action']")
            )
        ).click()
        time.sleep(3)
        print("  Dismissed consent popup.")
    except Exception:
        print("  No consent popup — continuing.")


def dismiss_google_signin_popup(driver):
    """
    FIX: Screenshots confirm a 'Sign in with Google' One-Tap overlay
    appears and blocks the Search button on attempts 2 and 3.
    This function aggressively removes it via multiple strategies.
    """
    # Strategy 1: switch into Google One-Tap iframe and click its close button
    try:
        iframe = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located(
                (By.XPATH, "//iframe[contains(@src,'accounts.google.com')]")
            )
        )
        driver.switch_to.frame(iframe)
        try:
            close_btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.ID, "close"))
            )
            close_btn.click()
            print("  Dismissed Google sign-in popup (iframe close).")
        except Exception:
            pass
        finally:
            driver.switch_to.default_content()
        time.sleep(0.8)
        return
    except Exception:
        driver.switch_to.default_content()

    # Strategy 2: click close button in the main document
    close_xpaths = [
        "//div[@id='credential_picker_container']//button",
        "//button[@aria-label='Close' or @aria-label='close']",
        "//*[@id='googleSignInOverlay']//button",
    ]
    for xpath in close_xpaths:
        try:
            btn = WebDriverWait(driver, 2).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            btn.click()
            print(f"  Dismissed Google sign-in popup (button).")
            time.sleep(0.8)
            return
        except Exception:
            continue

    # Strategy 3: Escape key
    try:
        ActionChains(driver).send_keys(Keys.ESCAPE).perform()
        time.sleep(0.5)
    except Exception:
        pass

    # Strategy 4: nuclear — remove all Google sign-in overlay elements from DOM
    try:
        removed = driver.execute_script("""
            let count = 0;
            const selectors = [
                '#credential_picker_container',
                'iframe[src*="accounts.google.com"]',
                '#googleSignInOverlay',
                'div[data-nosnippet]',
            ];
            selectors.forEach(sel => {
                document.querySelectorAll(sel).forEach(el => {
                    el.remove();
                    count++;
                });
            });
            return count;
        """)
        if removed:
            print(f"  Removed {removed} Google sign-in overlay element(s) via JS.")
        time.sleep(0.3)
    except Exception:
        pass


# ── 6. Location input ────────────────────────────────────────
def set_location(driver, location):
    """Type the pickup city into the From? box and confirm via autocomplete."""
    time.sleep(3)
    inp = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.XPATH, "//input[@placeholder='From?']"))
    )
    driver.execute_script("arguments[0].click();", inp)
    time.sleep(1)
    driver.execute_script("arguments[0].value = '';", inp)
    inp.send_keys(Keys.CONTROL + "a")
    inp.send_keys(Keys.DELETE)
    time.sleep(0.5)

    for char in location:
        inp.send_keys(char)
        time.sleep(0.05)
    time.sleep(2.5)

    try:
        first_suggestion = WebDriverWait(driver, 8).until(
            EC.element_to_be_clickable(
                (By.XPATH, "(//div[contains(@class,'autocomplete') or "
                           "contains(@class,'suggestion') or "
                           "contains(@class,'option')])[1]")
            )
        )
        first_suggestion.click()
        print("  Location set via autocomplete suggestion.")
    except Exception:
        inp.send_keys(Keys.RETURN)
        print("  Location set via Enter key.")
    time.sleep(2)


# ── 7. Date picking ──────────────────────────────────────────
def open_date_picker(driver, field="start"):
    """Open the start or end date picker."""
    selectors = []
    if field == "start":
        selectors = [
            "//div[@role='button' and @aria-label='Select start date from calendar input']",
            "//div[@role='button' and contains(@aria-label,'start date')]",
            "(//div[contains(@class,'c_iEC-datepicker')]//div[@role='button'])[1]",
            "(//div[contains(@class,'date')][@role='button'])[1]",
        ]
    else:
        selectors = [
            "//div[@role='button' and @aria-label='Select end date from calendar input']",
            "//div[@role='button' and contains(@aria-label,'end date')]",
            "(//div[contains(@class,'c_iEC-datepicker')]//div[@role='button'])[2]",
            "(//div[contains(@class,'date')][@role='button'])[2]",
        ]

    for xpath in selectors:
        try:
            btn = WebDriverWait(driver, 6).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.4)
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(1.5)
            print(f"  Opened {field} date picker.")
            return
        except Exception:
            continue

    raise Exception(f"Could not open {field} date picker.")


def pick_calendar_date(driver, date_obj):
    """
    FIX: Kayak's calendar cells have NO aria-label — they are plain divs
    containing only a day number. The correct strategy is to locate the
    month heading ('March 2026'), find its enclosing calendar panel,
    then click the cell whose text exactly equals the day number.
    """
    target_month = date_obj.strftime("%B %Y")   # e.g. "March 2026"
    day_num      = str(date_obj.day)             # e.g. "22"
    aria_label   = format_kayak_aria(date_obj)
    date_str     = date_obj.strftime("%Y-%m-%d")

    print(f"  Picking {day_num} in '{target_month}'...")

    # Attempt 1: aria-label (works on some Kayak variants)
    for xpath in [
        f"//*[@aria-label='{aria_label}']",
        f"//div[@role='button' and @aria-label='{aria_label}']",
    ]:
        try:
            cell = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            driver.execute_script("arguments[0].click();", cell)
            print(f"  Selected via aria-label.")
            time.sleep(1)
            return
        except Exception:
            continue

    # Attempt 2: data-date / data-value attributes
    for xpath in [
        f"//*[@data-date='{date_str}']",
        f"//*[@data-value='{date_str}']",
    ]:
        try:
            cell = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            driver.execute_script("arguments[0].click();", cell)
            print(f"  Selected via data attribute.")
            time.sleep(1)
            return
        except Exception:
            continue

    # Attempt 3 (PRIMARY FIX): month-heading + day-number text matching
    _click_day_by_month_heading(driver, date_obj)


def _click_day_by_month_heading(driver, date_obj, max_nav=6):
    """
    FIX: Core fix for Kayak's no-aria-label calendar cells.

    Kayak shows two month panels side by side. Each has a heading div
    ('March 2026', 'April 2026'). Day cells are plain divs whose only
    content is the day number. Algorithm:
      1. Find the heading whose text == target_month.
      2. Walk up ancestor elements to find the enclosing month panel.
      3. Inside that panel, find the div/td with text == day_num that
         is visible and not disabled.
      4. If month not yet visible, click the '>' Next Month button.
    """
    target_month = date_obj.strftime("%B %Y")
    day_num      = str(date_obj.day)

    for nav_attempt in range(max_nav + 1):
        # ── Step 1: locate the month heading ─────────────────
        month_heading = None
        for hx in [
            f"//*[normalize-space(text())='{target_month}']",
            f"//*[contains(text(),'{target_month}')]",
        ]:
            try:
                month_heading = WebDriverWait(driver, 4).until(
                    EC.presence_of_element_located((By.XPATH, hx))
                )
                break
            except Exception:
                continue

        if month_heading:
            # ── Step 2 & 3: find enclosing panel, then day cell ──
            for ancestor_depth in range(1, 7):
                ancestor_path = "/".join([".."] * ancestor_depth)
                try:
                    container = month_heading.find_element(By.XPATH, ancestor_path)
                except Exception:
                    continue

                for cx in [
                    f".//*[self::div or self::td or self::button]"
                    f"[normalize-space(text())='{day_num}']",
                    f".//*[self::div or self::td or self::button]"
                    f"[./span[normalize-space(text())='{day_num}']]",
                ]:
                    try:
                        candidates = container.find_elements(By.XPATH, cx)
                        clickable = [
                            c for c in candidates
                            if c.is_displayed()
                            and c.is_enabled()
                            and "disabled" not in (c.get_attribute("class") or "").lower()
                            and "past" not in (c.get_attribute("class") or "").lower()
                        ]
                        if clickable:
                            cell = clickable[0]
                            driver.execute_script(
                                "arguments[0].scrollIntoView({block:'center'});", cell
                            )
                            time.sleep(0.3)
                            driver.execute_script("arguments[0].click();", cell)
                            print(
                                f"  Clicked day {day_num} in '{target_month}' "
                                f"(ancestor depth {ancestor_depth})."
                            )
                            time.sleep(1)
                            return
                    except Exception:
                        continue

        if nav_attempt == max_nav:
            break

        # ── Step 4: click the Next Month '>' button ───────────
        next_xpaths = [
            "//button[@aria-label='Next month']",
            "//div[@aria-label='Next month']",
            "//*[contains(@aria-label,'Next month')]",
            "//*[contains(@aria-label,'next month')]",
            # FIX: the '>' arrow visible in screenshots
            "(//div[contains(@class,'calendar') or "
            "contains(@class,'c_iEC')]//button)[last()]",
            "(//div[contains(@class,'Calendar')]//button)[last()]",
        ]
        navigated = False
        for nx in next_xpaths:
            try:
                btn = WebDriverWait(driver, 3).until(
                    EC.element_to_be_clickable((By.XPATH, nx))
                )
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1)
                navigated = True
                print(f"  Advanced calendar month (attempt {nav_attempt + 1}).")
                break
            except Exception:
                continue

        if not navigated:
            print("  Could not find Next Month button — stopping navigation.")
            break

    raise Exception(
        f"Could not click day {day_num} in '{target_month}' "
        f"after {max_nav} navigation attempts."
    )


def set_dates(driver, start_date, end_date):
    """
    Full date-setting flow.
    FIX: Explicitly close the calendar overlay after both dates are set
    so it doesn't block the Search button.
    """
    open_date_picker(driver, field="start")
    time.sleep(0.8)
    pick_calendar_date(driver, start_date)
    time.sleep(0.8)

    # Kayak usually auto-advances to end selection; only re-open if needed
    if not _is_calendar_open(driver):
        open_date_picker(driver, field="end")
        time.sleep(0.8)

    pick_calendar_date(driver, end_date)
    time.sleep(0.8)

    # FIX: close the calendar before clicking Search
    _close_calendar(driver)
    time.sleep(0.5)

    _verify_dates(driver, start_date, end_date)


def _is_calendar_open(driver):
    """Return True if a calendar overlay is currently visible."""
    for xpath in [
        "//div[contains(@class,'c_iEC-calendar')]",
        "//*[contains(@class,'DayPicker') and not(contains(@style,'display:none'))]",
        # FIX: detect by presence of month headings (as seen in screenshots)
        "//div[.//*[normalize-space(text())='March 2026'] or "
        ".//*[normalize-space(text())='April 2026']]",
    ]:
        try:
            el = driver.find_element(By.XPATH, xpath)
            if el.is_displayed():
                return True
        except Exception:
            continue
    return False


def _close_calendar(driver):
    """
    FIX: After setting dates Kayak leaves the calendar open, which can
    intercept pointer events on the Search button. Close it first.
    """
    if not _is_calendar_open(driver):
        return
    # Try Escape
    try:
        ActionChains(driver).send_keys(Keys.ESCAPE).perform()
        time.sleep(0.6)
        if not _is_calendar_open(driver):
            print("  Calendar closed via Escape.")
            return
    except Exception:
        pass
    # Click outside the calendar (top-left of viewport)
    try:
        ActionChains(driver).move_by_offset(10, 10).click().perform()
        time.sleep(0.5)
        if not _is_calendar_open(driver):
            print("  Calendar closed via outside click.")
            return
    except Exception:
        pass
    # JS body click
    try:
        driver.execute_script("document.body.dispatchEvent(new MouseEvent('click'));")
        time.sleep(0.4)
        print("  Calendar closed via JS body click.")
    except Exception:
        pass


def _verify_dates(driver, start_date, end_date):
    """Log what the date buttons display after selection."""
    for label in ["start date", "end date"]:
        try:
            btn = driver.find_element(
                By.XPATH, f"//div[@role='button' and contains(@aria-label,'{label}')]"
            )
            print(f"  {label.capitalize()} shows: {btn.text.strip()!r}")
        except Exception:
            print(f"  Could not verify {label} field (non-fatal).")


# ── 8. Search button ─────────────────────────────────────────
def perform_search(driver):
    """
    FIX: Two-stage approach:
      1. Aggressively dismiss the Google One-Tap popup (seen in screenshots).
      2. Click the Search button using multiple selector fallbacks.
    """
    # Always dismiss before trying to click Search
    dismiss_google_signin_popup(driver)
    time.sleep(0.5)

    search_xpaths = [
        "//button[@aria-label='Click to search cars']",
        "//button[contains(@class,'search') and contains(@class,'submit')]",
        "//button[contains(@class,'c_iEC-btn') and contains(@class,'submit')]",
        "//div[@class='a7Uc-infix']//button",
        "//div[contains(@class,'a7Uc')]//button",
        "//button[normalize-space(text())='Search']",
        "//button[.//span[normalize-space(text())='Search']]",
    ]

    for xpath in search_xpaths:
        try:
            btn = WebDriverWait(driver, 8).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.4)

            # Dismiss one final time immediately before clicking
            dismiss_google_signin_popup(driver)
            time.sleep(0.3)

            try:
                btn.click()
            except Exception:
                # ElementClickInterceptedException — force via JS
                driver.execute_script("arguments[0].click();", btn)

            print("  Search triggered.")

            # Wait for results URL rather than blind sleep
            WebDriverWait(driver, 20).until(
                lambda d: "cars" in d.current_url
                and d.current_url != "https://www.kayak.com/cars"
            )
            time.sleep(5)
            return
        except Exception:
            continue

    raise Exception("Could not locate or click the Search button.")


# ── 9. Results tab & loading ─────────────────────────────────
def switch_to_results_tab(driver):
    try:
        WebDriverWait(driver, 20).until(lambda d: len(d.window_handles) > 1)
        driver.switch_to.window(driver.window_handles[1])
        print("  Switched to results tab.")
        time.sleep(5)
        return True
    except Exception:
        print("  No new tab — continuing in current tab.")
        return False


def load_all_results(driver):
    clicks = 0
    while True:
        try:
            btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//div[contains(text(),'Show more results')]")
                )
            )
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(1)
            btn.click()
            clicks += 1
            print(f"  Loaded more results (click {clicks})")
            time.sleep(5)
        except Exception:
            print(f"  All results loaded ({clicks} extra page(s)).")
            break


# ── 10. Data extraction & persistence ────────────────────────

# ---------------------------------------------------------------------------
# SELECTOR CATALOGUE
# Kayak uses obfuscated, deploy-time CSS class names (e.g. "c4nz8-price-total",
# "MseY-title") that change without notice.  Every list below provides:
#   • a structural / semantic XPath that works regardless of class names, AND
#   • the legacy class-name selector as a fallback (kept for continuity).
# The helper _find_all() tries each selector in order and returns the first
# non-empty result, so the scraper degrades gracefully when Kayak redeploys.
# ---------------------------------------------------------------------------

def _find_all(driver, xpath_list, timeout=20):
    """
    Try each XPath in xpath_list.  Return the first list that is non-empty.
    Waits up to `timeout` seconds on the first selector; remaining selectors
    get a short 2-second probe so we don't stall the whole run.
    """
    for i, xpath in enumerate(xpath_list):
        wait_secs = timeout if i == 0 else 2
        try:
            WebDriverWait(driver, wait_secs).until(
                EC.presence_of_element_located((By.XPATH, xpath))
            )
            els = driver.find_elements(By.XPATH, xpath)
            if els:
                print(f"    [selector hit] {xpath[:80]}")
                return els
        except Exception:
            continue
    return []


def _dump_page_structure(driver, start_date):
    """
    Save a DOM snapshot when extraction finds 0 results.
    Helps diagnose selector drift without having to re-run.
    """
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        path = os.path.join(
            OUTPUT_DIR,
            f"dom_dump_{start_date.strftime('%Y%m%d')}_{datetime.now().strftime('%H%M%S')}.html"
        )
        with open(path, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"  DOM snapshot saved → {path}  (open in browser to inspect selectors)")
    except Exception as e:
        print(f"  Could not save DOM snapshot: {e}")


# ── Selector lists (structural first, legacy class-name second) ──────────────

# Each car listing is wrapped in a result card. All other selectors are scoped
# relative to these cards so mismatched list lengths become impossible.
CARD_SELECTORS = [
    # Structural: a result card is a list-item or article containing both a
    # price element and a car-name element
    "//div[contains(@class,'result') and .//*[contains(@class,'price')]]",
    "//div[contains(@class,'Result') and .//*[contains(@class,'price')]]",
    "//li[contains(@class,'result')]",
    # Legacy class name (Kayak 2024–2025)
    "//*[contains(@class,'yuAt-result') or contains(@class,'Fxw9')]",
    # Broadest fallback: any div that contains both a dollar-sign price text
    # and a car-name string
    "//div[.//*[contains(text(),'$')] and .//*[contains(@class,'title') "
    "or contains(@class,'name') or contains(@class,'car')]]",
]

PRICE_SELECTORS_IN_CARD = [
    # Structural: element containing a '$' followed by digits
    ".//*[contains(text(),'$') and string-length(normalize-space(text())) < 15]",
    # Common Kayak price class patterns
    ".//*[contains(@class,'price-total') or contains(@class,'Price-total')]",
    ".//*[contains(@class,'price') and contains(@class,'total')]",
    ".//*[contains(@class,'price') and not(contains(@class,'per'))]",
    # Legacy
    ".//*[contains(@class,'c4nz8-price-total')]",
]

CAR_NAME_SELECTORS_IN_CARD = [
    # Structural: heading-level element or element with 'title'/'name' in class
    ".//*[self::h2 or self::h3 or self::h4][normalize-space(text())!='']",
    ".//*[contains(@class,'title') and contains(@class,'js-title')]",
    ".//*[contains(@class,'title') and not(contains(@class,'sub'))]",
    ".//*[contains(@class,'name') and not(contains(@class,'agency') "
    "or contains(@class,'provider'))]",
    # Legacy
    ".//*[contains(@class,'MseY-title')]",
]

CAR_DETAILS_SELECTORS_IN_CARD = [
    # Structural: a subtitle / specs row below the main car name
    ".//*[contains(@class,'sub') and (contains(@class,'title') "
    "or contains(@class,'detail') or contains(@class,'spec'))]",
    ".//*[contains(@class,'details') or contains(@class,'features')]",
    # Legacy
    ".//*[contains(@class,'MseY-sub-title')]",
]

LOCATION_SELECTORS_IN_CARD = [
    # Structural: element containing typical pickup-location text patterns
    ".//*[contains(@class,'location') or contains(@class,'pickup') "
    "or contains(@class,'address')]",
    ".//*[contains(@class,'row') and contains(@class,'first')]",
    # Legacy
    ".//*[contains(@class,'G0cp-row') and contains(@class,'G0cp-first-row')]",
    ".//*[contains(@class,'G0cp')]",
]

PROVIDER_IMG_SELECTORS_IN_CARD = [
    # Structural: img inside an agency/provider/logo container
    ".//img[contains(@class,'agency') or contains(@class,'provider') "
    "or contains(@class,'logo')]",
    ".//img[contains(@alt,'Hertz') or contains(@alt,'Avis') or contains(@alt,'Budget') "
    "or contains(@alt,'Enterprise') or contains(@alt,'National') "
    "or contains(@alt,'Alamo') or contains(@alt,'Dollar') "
    "or contains(@alt,'Thrifty') or contains(@alt,'Sixt') "
    "or contains(@alt,'Europcar')]",
    # Any img inside an element with 'agency' in its class
    ".//*[contains(@class,'agency') or contains(@class,'vendor')]//img",
    # Legacy
    ".//*[contains(@class,'mR2O-agency-image')]//img",
    # Fallback: any img in the card
    ".//img[@alt or @src]",
]


def _text_from_card(card, selectors):
    """Return the stripped text of the first matching element inside `card`."""
    for xpath in selectors:
        try:
            el = card.find_element(By.XPATH, xpath)
            txt = el.text.strip()
            if txt:
                return txt
        except Exception:
            continue
    return ""


def _html_from_card(card, selectors):
    """Return the outerHTML of the first matching element inside `card`."""
    for xpath in selectors:
        try:
            el = card.find_element(By.XPATH, xpath)
            return el.get_attribute("outerHTML") or ""
        except Exception:
            continue
    return ""


def _provider_from_card(card):
    """Return provider name (alt) or image src from the first img hit."""
    for xpath in PROVIDER_IMG_SELECTORS_IN_CARD:
        try:
            img = card.find_element(By.XPATH, xpath)
            return img.get_attribute("alt") or img.get_attribute("src") or ""
        except Exception:
            continue
    return ""


def extract_data(driver, start_date, end_date):
    """
    Extract car rental listings from the loaded results page.

    Strategy:
      1. Wait for at least one result card to appear.
      2. Collect all result cards.
      3. For each card, extract fields using the selector catalogues above.
         This keeps every field aligned — mismatched list lengths are
         impossible because we iterate over cards, not parallel lists.
      4. If 0 cards found, dump the DOM for inspection and return [].
    """
    print("  Waiting for result cards...")

    # ── Wait for cards ────────────────────────────────────────
    cards = []
    for xpath in CARD_SELECTORS:
        try:
            WebDriverWait(driver, 25).until(
                EC.presence_of_element_located((By.XPATH, xpath))
            )
            cards = driver.find_elements(By.XPATH, xpath)
            if cards:
                print(f"  Found {len(cards)} result card(s) via: {xpath[:70]}")
                break
        except Exception:
            continue

    if not cards:
        print("  WARNING: No result cards found — dumping DOM for inspection.")
        _dump_page_structure(driver, start_date)
        return []

    # ── Extract per card ──────────────────────────────────────
    results = []
    skipped = 0
    for card in cards:
        price    = _text_from_card(card, PRICE_SELECTORS_IN_CARD)
        name     = _text_from_card(card, CAR_NAME_SELECTORS_IN_CARD)
        details  = _html_from_card(card, CAR_DETAILS_SELECTORS_IN_CARD)
        location = _text_from_card(card, LOCATION_SELECTORS_IN_CARD)
        provider = _provider_from_card(card)

        # Skip cards where both name and price are empty (likely ad / banner)
        if not name and not price:
            skipped += 1
            continue

        results.append({
            "Name"           : name,
            "Name_Element"   : BeautifulSoup(details, "html.parser").get_text(strip=True) if details else "",
            "Location"       : location,
            "Price"          : price,
            "Provider"       : provider,
            "Start_Date"     : start_date.strftime("%Y-%m-%d"),
            "End_Date"       : end_date.strftime("%Y-%m-%d"),
            "Extraction_Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })

    print(f"  Extracted {len(results)} records ({skipped} card(s) skipped).")

    # ── Warn if fields look sparse — points to selector drift ─
    empty_prices = sum(1 for r in results if not r["Price"])
    empty_names  = sum(1 for r in results if not r["Name"])
    if empty_prices > len(results) * 0.5 or empty_names > len(results) * 0.5:
        print(
            f"  WARNING: >50 % of records have empty Price ({empty_prices}) "
            f"or Name ({empty_names}). "
            f"Kayak may have changed its DOM — check the DOM snapshot."
        )
        _dump_page_structure(driver, start_date)

    return results


def save_to_csv(data, location, start_date, end_date):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    slug     = location.split(",")[0].replace(" ", "_")
    filename = (
        f"{OUTPUT_DIR}/kayak_{slug}"
        f"_{start_date.strftime('%Y%m%d')}_to_{end_date.strftime('%Y%m%d')}"
        f"_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )
    fieldnames = ["Name", "Name_Element", "Location", "Price", "Provider",
                  "Start_Date", "End_Date", "Extraction_Date"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    print(f"  Saved {len(data)} records → {filename}")
    return filename


# ── 11. Single-day scrape with retry ─────────────────────────
def scrape_single_day(driver, location, start_date, end_date, attempt=1):
    try:
        driver.get("https://www.kayak.com/cars")
        time.sleep(5)
        handle_initial_popups(driver)
        set_location(driver, location)
        set_dates(driver, start_date, end_date)
        perform_search(driver)
        switch_to_results_tab(driver)
        load_all_results(driver)
        results = extract_data(driver, start_date, end_date)
        if results:
            return save_to_csv(results, location, start_date, end_date)
        print(f"  No results for {start_date.date()}.")
        return None
    except Exception as exc:
        try:
            driver.save_screenshot(f"error_{start_date.strftime('%Y%m%d')}_attempt{attempt}.png")
        except Exception:
            pass
        print(f"  ERROR attempt {attempt}: {exc}")
        return None


# ── 12. Main loop ─────────────────────────────────────────────
def main():
    date_range = get_date_range(START_DATE_STR, END_DATE_STR)
    if not date_range:
        print("Invalid date range.")
        return []

    print(f"\nDriveSmart scrape | {LOCATION}")
    print(f"  {START_DATE_STR} → {END_DATE_STR} ({len(date_range)} days) | browser: {BROWSER}\n")

    driver      = setup_driver()
    saved_files = []

    try:
        for i, current_date in enumerate(date_range):
            next_date = current_date + timedelta(days=1)
            print(f"\n[{i+1}/{len(date_range)}] {current_date.date()} → {next_date.date()}")

            for attempt in range(1, MAX_RETRIES + 1):
                fname = scrape_single_day(driver, LOCATION, current_date, next_date, attempt)
                if fname:
                    saved_files.append(fname)
                    break
                if attempt < MAX_RETRIES:
                    wait = 2 ** attempt
                    print(f"  Retrying in {wait}s...")
                    time.sleep(wait)
            else:
                print(f"  Skipped {current_date.date()} after {MAX_RETRIES} attempts.")

            while len(driver.window_handles) > 1:
                driver.switch_to.window(driver.window_handles[-1])
                driver.close()
            driver.switch_to.window(driver.window_handles[0])

            if i < len(date_range) - 1:
                time.sleep(DELAY_BETWEEN_DAYS + random.uniform(0, 2))

    finally:
        driver.quit()
        print(f"\nDone. {len(saved_files)}/{len(date_range)} days saved.")

    return saved_files


# ── 13. Merge CSVs into master dataset ───────────────────────
def merge_csv_files(output_dir=OUTPUT_DIR, master_name="kayak_master.csv"):
    files = sorted(glob.glob(os.path.join(output_dir, "kayak_*.csv")))
    if not files:
        print(f"No CSV files found in '{output_dir}'.")
        return pd.DataFrame()

    print(f"Merging {len(files)} file(s)...")
    master = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    before = len(master)
    master.drop_duplicates(
        subset=["Name", "Location", "Price", "Provider", "Start_Date"], inplace=True
    )
    print(f"Rows: {before} → {len(master)} (removed {before - len(master)} dupes)")
    path = os.path.join(output_dir, master_name)
    master.to_csv(path, index=False)
    print(f"Master CSV → {path}")
    return master


# ── 14. Entry point ──────────────────────────────────────────
if __name__ == "__main__":
    saved = main()
    df    = merge_csv_files()
    if not df.empty:
        print(df.shape)
        print(df.head())


DriveSmart scrape | San Jose, California, United States
  2026-03-22 → 2026-03-29 (8 days) | browser: chrome


[1/8] 2026-03-22 → 2026-03-23
  Dismissed consent popup.
  Location set via Enter key.
  Opened start date picker.
  Picking 22 in 'March 2026'...
  Clicked day 22 in 'March 2026' (ancestor depth 1).
  Picking 23 in 'March 2026'...
  Clicked day 23 in 'March 2026' (ancestor depth 1).
  Start date shows: 'Sun 3/22'
  End date shows: 'Mon 3/23'
  Search triggered.
  Switched to results tab.
  Loaded more results (click 1)
  Loaded more results (click 2)
  Loaded more results (click 3)
  Loaded more results (click 4)
  Loaded more results (click 5)
  Loaded more results (click 6)
  Loaded more results (click 7)
  Loaded more results (click 8)
  Loaded more results (click 9)
  Loaded more results (click 10)
  Loaded more results (click 11)
  Loaded more results (click 12)
  Loaded more results (click 13)
  Loaded more results (click 14)
  Loaded more results (click 15)
  Loaded m

KeyboardInterrupt: 